In [36]:
import pandas as pd
import json
import numpy as np

excel_file = 'data/MASTERDATA.xlsx'
json_master_file = 'cleaned_master_data.json'

print("dang doc du lieu...")
with open(json_master_file, 'r', encoding='utf-8') as f:
    master_data = json.load(f)

df_plan = pd.read_excel(excel_file, sheet_name='KẾ HOẠCH SẢN XUẤT', header=2)
df_specs = pd.read_excel(excel_file, sheet_name='BUILD APPSHEET', header=3)

df_plan_clean = df_plan[['MÃ NHÀ MÁY', 'SL', 'Ngày hoàn thành', 'NỘI DUNG CÔNG VIỆC']].copy()
df_plan_clean.columns = ['job_id', 'quantity', 'due_date', 'status']

# df_plan_clean = df_plan_clean[df_plan_clean['status'] == 'PC'] # Bỏ comment nếu muốn lọc


df_specs_clean = df_specs[['Mã nhà máy', 'Vật liệu gia công', 'Số mét dài (m)', 'Kích thước DxR (mm)']].copy()
df_specs_clean.columns = ['job_id', 'material_code', 'cutting_length_m', 'dimension_str']


df_merged = pd.merge(df_plan_clean, df_specs_clean, on='job_id', how='inner')
print(f"-> Tìm thấy {len(df_merged)} đơn hàng có đầy đủ thông tin.")
print(df_specs_clean)

dang doc du lieu...
-> Tìm thấy 2 đơn hàng có đầy đủ thông tin.
          job_id  material_code  cutting_length_m  dimension_str
0            NaN   100000470839               NaN            NaN
1   1.004240e+12   100000447257               3.0          300.0
2   1.001241e+12   100000420858               4.0         1800.0
3            NaN   100000470512               NaN            NaN
4            NaN   100000466103               NaN            NaN
5            NaN   100000466477               NaN            NaN
6            NaN   100000470514               NaN            NaN
7            NaN   100000470498               NaN            NaN
8            NaN   100000474255               NaN            NaN
9            NaN   100000470834               NaN            NaN
10           NaN        Polaris               NaN            NaN
11           NaN  Solid surface               NaN            NaN
12           NaN    Tundar grey               NaN            NaN
13           NaN        Vo

In [37]:
# --- 3. HÀM TÍNH TOÁN THỜI GIAN ---

def get_size_category(dim_str):
    """
    Phân loại kích thước dựa trên chuỗi '600x600'
    Logic: Lấy cạnh NHỎ NHẤT để quyết định (vì cạnh nhỏ khó cắt nhanh hơn)
    """
    try:
        if pd.isna(dim_str): return "> 600 mm" # Mặc định
        # Tách chuỗi 600x600 hoặc 600 x 600
        parts = str(dim_str).lower().replace('mm','').split('x')
        dims = [float(p.strip()) for p in parts if p.strip().replace('.','').isdigit()]
        if not dims: return "> 600 mm"

        min_dim = min(dims)

        # Mapping theo bảng NANG_SUAT của bạn
        if min_dim < 200: return "< 200 mm"
        elif 200 <= min_dim < 400: return "200- 400 mm"
        elif 400 <= min_dim < 600: return "400-600 mm"
        else: return "> 600 mm"
    except:
        return "> 600 mm"

def calculate_processing_times(row):
    """
    Tính thời gian chạy trên TẤT CẢ các máy có thể
    """
    times = {}

    # 1. Xác định Nhóm năng suất (A, B, C...)
    mat_id = str(row['material_code'])
    # Tra cứu trong master_data, nếu không thấy thì mặc định là 'A'
    group = master_data['materials_map'].get(mat_id, 'A')

    # 2. Xác định Loại kích thước
    size_cat = get_size_category(row['dimension_str'])

    # 3. Tính Tổng mét dài cần cắt (mét -> mm)
    # Công thức: (Mét dài 1 cái * Số lượng) * 1000
    try:
        total_length_mm = float(row['cutting_length_m']) * float(row['quantity']) * 1000
    except:
        return {} # Lỗi dữ liệu thì bỏ qua

    if total_length_mm <= 0: return {}

    # 4. Duyệt qua từng máy để tính thời gian
    for machine_id, machine_data in master_data['machines'].items():
        # Kiểm tra máy có chức năng "Cắt thẳng" hoặc "Cut To Size" không?
        # (Bạn có thể in machine_data['capabilities'] ra để xem tên chính xác)
        caps = machine_data.get('capabilities', [])
        is_capable = any("Cắt thẳng" in c or "Cut To Size" in c for c in caps)
        print(caps)
        if is_capable:
            # Tra tốc độ
            try:
                speed = machine_data['speed_matrix'][group][size_cat]
                if speed > 0:
                    # Thời gian (phút) = Quãng đường / Vận tốc
                    t_min = total_length_mm / speed
                    times[machine_id] = round(t_min, 2)
            except:
                pass # Không có dữ liệu tốc độ cho nhóm này thì bỏ qua

    return times

In [38]:
final_jobs = []

for index, row in df_merged.iterrows():
    # Tính P_ij
    proc_times = calculate_processing_times(row)

    # Chỉ thêm vào danh sách nếu tìm thấy ít nhất 1 máy làm được
    if proc_times:
        job_data = {
            "id": str(row['job_id']),
            "quantity": int(row['quantity']),
            "due_date": str(row['due_date']), # Lưu ý: Cần xử lý datetime sau này
            "material_group": master_data['materials_map'].get(str(row['material_code']), 'Unknown'),
            "processing_times": proc_times # Đây chính là P_ij
        }
        final_jobs.append(job_data)

['Cắt thẳng']
['Cắt thẳng', 'Chém 45', 'Phào chỉ (CMS)']
['Cắt thẳng', 'Cắt dị hình']
[]
['Cắt thẳng', 'Chém 45', 'Phào chỉ (CMS)']
['Chém 45', 'Phào chỉ (CMS)', 'VÁT LÁ HẸ', 'Đánh bóng ']
['Cắt thẳng', 'Chém 45']
['Cắt thẳng', 'Cắt dị hình']
['Cắt thẳng', 'Cắt dị hình']
[]
['Cắt thẳng', 'Cắt dị hình']
['Đánh bóng ']
['Cắt thẳng', 'Cắt dị hình']
['Đánh bóng ']
['VÁT LÁ HẸ']
['Cắt thẳng', 'Cắt dị hình']
['Chém 45']
['Chém 45', 'Phào chỉ (CMS)', 'VÁT LÁ HẸ', 'Đánh bóng ']
['Cắt thẳng']
['Cắt thẳng', 'Chém 45', 'Phào chỉ (CMS)']
['Cắt thẳng', 'Cắt dị hình']
[]
['Cắt thẳng', 'Chém 45', 'Phào chỉ (CMS)']
['Chém 45', 'Phào chỉ (CMS)', 'VÁT LÁ HẸ', 'Đánh bóng ']
['Cắt thẳng', 'Chém 45']
['Cắt thẳng', 'Cắt dị hình']
['Cắt thẳng', 'Cắt dị hình']
[]
['Cắt thẳng', 'Cắt dị hình']
['Đánh bóng ']
['Cắt thẳng', 'Cắt dị hình']
['Đánh bóng ']
['VÁT LÁ HẸ']
['Cắt thẳng', 'Cắt dị hình']
['Chém 45']
['Chém 45', 'Phào chỉ (CMS)', 'VÁT LÁ HẸ', 'Đánh bóng ']


In [39]:
with open('ready_to_optimize_jobs.json', 'w', encoding='utf-8') as f:
    json.dump(final_jobs, f, ensure_ascii=False, indent=4)

print(f"HOÀN TẤT! Đã tạo file 'ready_to_optimize_jobs.json' với {len(final_jobs)} đơn hàng sẵn sàng chạy thuật toán.")
# Xem thử 1 đơn hàng mẫu
if final_jobs:
    print("Mẫu dữ liệu Job đầu tiên:")
    print(json.dumps(final_jobs[1], indent=2, ensure_ascii=False))

HOÀN TẤT! Đã tạo file 'ready_to_optimize_jobs.json' với 2 đơn hàng sẵn sàng chạy thuật toán.
Mẫu dữ liệu Job đầu tiên:
{
  "id": "1001240606311",
  "quantity": 1,
  "due_date": "2024-10-10 00:00:00",
  "material_group": "B",
  "processing_times": {
    "CNCCCC0003": 1.82,
    "CNCTNC0002": 10.0,
    "CNCCCC0002": 1.82,
    "CNCCCC0001": 2.5,
    "CNCTNC0005": 10.0,
    "CNCTNC0004": 10.0,
    "CNCTNC0003": 10.0,
    "CNCTNC0001": 10.0
  }
}
